<a href="https://colab.research.google.com/github/Meghna2706/IN226095202_GEN_AI/blob/main/Task3_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ASSIGNMENT NLP – 3 (Chatbot using Transformers)

## Assignment: Build a Chatbot using Hugging Face Transformers

**Objective**

Build a simple conversational chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.


### What are transformers?

A Transformers are deep learning model that understands text by looking at all words at the same time and figuring out how they relate to each other.

* Old models (RNN/LSTM): read word-by-word that might forget earlier context

* Transformer: looks at all words together, compares them and figures out

---

**Key Concepts**

| Concept         | Simple Description                                      |
|----------------|----------------------------------------------------------|
| Tokenization   | Break text into small pieces (words/tokens)              |
| Embeddings     | Convert words into meaningful numbers                    |
| Self-Attention | Words look at each other to understand context           |
| Causal LM      | Predict the next word using previous words               |
| Pre-training   | Model learns from large text data before actual use      |

---

### DialoGPT

* DialoGPT is a chatbot model built by Microsoft.
* It is designed specifically to have conversations like a human.
* Available in 3 sizes: small, medium, large — we use medium for better quality
---
### Hugging Face
Hugging Face is the leading open-source AI platform that provides:

* Thousands of pre-trained models ready to use in just a few lines of code
* The transformers Python library — a unified API for 100+ model architectures
* A Model Hub at huggingface.co where researchers share models publicly
---
**Chatbot Workflow**

User Input → Model Processing → Response Generation → Display Output → Loop Until Exit
---

**1. Model Loading (Mandatory)**

In [ ]:
!pip install transformers torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
model_name = "microsoft/DialoGPT-small"  # fast & stable

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
model.eval()

print("Model loaded successfully!")

Model loaded successfully!


**2. Response Generation Function**

In [ ]:
def generate_response(user_input, chat_history_ids):

    # Rule-based responses
    text = user_input.lower().strip()

    if text in ["hi", "hello", "hey"]:
        return "Hello! Nice to meet you. How can I assist you today?", chat_history_ids

    elif "artificial intelligence" in text or "what is ai" in text:
        return "Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.", chat_history_ids

    elif "who created python" in text:
        return "Python was created by Guido van Rossum and released in 1991.", chat_history_ids

    elif "thank you" in text or "thanks" in text:
        return "You're welcome! Feel free to ask more questions.", chat_history_ids

    # Encode user input correctly (NO custom prompt)
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    )

    # Add history properly
    if chat_history_ids is not None:
        input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        input_ids,
        max_length=input_ids.shape[-1] + 50,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,          # allow natural replies
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    # Extract only new response
    response = tokenizer.decode(
        chat_history_ids[:, input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

**3. Continuous Conversation**

In [ ]:
def run_chatbot():

    chat_history_ids = None

    print("-" * 60)
    print("🤖 AI Chatbot Started")
    print("Type 'exit' or 'quit' to stop")
    print("-" * 60)

    print("\nChatbot: Hello! I am your AI assistant. How can I help you today?\n")

    while True:
        user_input = input("User: ").strip()

        # Handle empty input
        if not user_input:
            print("Chatbot: Please enter something.")
            continue

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! 👋")
            break

        # Generate response
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        print("Chatbot:", response)
        print()


# Run chatbot
if __name__ == "__main__":
    run_chatbot()

------------------------------------------------------------
🤖 AI Chatbot Started
Type 'exit' or 'quit' to stop
------------------------------------------------------------

Chatbot: Hello! I am your AI assistant. How can I help you today?

User: Hello
Chatbot: Hello! Nice to meet you. How can I assist you today?

User: What is Artificial Intelligence?
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.

User: Who created Python?
Chatbot: Python was created by Guido van Rossum and released in 1991.

User: Thank you
Chatbot: You're welcome! Feel free to ask more questions.

User: exit
Chatbot: Goodbye! 👋


## Features Implemented

* Uses Hugging Face transformer model
* Dynamic response generation (no hardcoding)
* Maintains conversation context
* Handles multiple user inputs
* Includes exit condition
---

**Conclusion**

- In this assignment, I successfully built a context-aware conversational chatbot using DialoGPT.
- The chatbot demonstrates how transformer models can generate human-like responses and maintain conversation flow.